# Create UC Metric Views (semantic layer)

Post-gold step (idempotent `CREATE OR REPLACE`). Defines governed **UC Metric Views**
over the same gold star schema that the pre-aggregated MVs use, so the sample shows both
options side by side:

- **Pre-aggregated MVs** (`fct_sales_by_month_segment`, `fct_product_performance`, ...) materialize a *fixed grain*.
- **Metric Views** below define the metric *semantics once* and can be queried at *any grain* via `MEASURE(...)`.

Run as a job task after the gold pipeline. Requires the gold tables (`fct_order_lines`,
`fct_part_supply`, `dim_*`) to already exist.

In [ ]:
dbutils.widgets.text("catalog", "main")
dbutils.widgets.text("schema_namespace", "tpch_sample")
dbutils.widgets.text("logical_env", "")

catalog = dbutils.widgets.get("catalog")
schema_namespace = dbutils.widgets.get("schema_namespace")
logical_env = dbutils.widgets.get("logical_env")

gold_schema = f"{schema_namespace}_gold{logical_env}"
g = f"{catalog}.{gold_schema}"

# Metric View 1 — sales / order-line semantics over fct_order_lines + conformed dims.
# Dimensions join on the surrogate keys the fact already resolved point-in-time, so attributes
# (e.g. market segment) reflect the dimension version effective as of the order date.
sales_metrics_yaml = f"""
version: 0.1
source: {g}.fct_order_lines
joins:
  - name: cust
    source: {g}.dim_customer
    on: source.customer_sk = cust.customer_sk
  - name: prod
    source: {g}.dim_part
    on: source.part_sk = prod.part_sk
  - name: dt
    source: {g}.dim_date
    on: source.order_date_key = dt.date_key
dimensions:
  - name: Order Year
    expr: dt.year
  - name: Order Quarter
    expr: dt.quarter
  - name: Order Month
    expr: dt.month
  - name: Order Date
    expr: source.order_date
  - name: Region
    expr: cust.region_name
  - name: Nation
    expr: cust.nation_name
  - name: Market Segment
    expr: cust.market_segment
  - name: Brand
    expr: prod.brand
  - name: Part Type
    expr: prod.part_type
  - name: Ship Mode
    expr: source.ship_mode
  - name: Order Priority
    expr: source.order_priority
  - name: Return Flag
    expr: source.return_flag
measures:
  - name: Gross Sales
    expr: SUM(source.extended_price)
  - name: Net Sales
    expr: SUM(source.net_sales)
  - name: Units Sold
    expr: SUM(source.quantity)
  - name: Order Count
    expr: COUNT(DISTINCT source.order_key)
  - name: Line Count
    expr: COUNT(1)
  - name: Active Customers
    expr: COUNT(DISTINCT source.customer_key)
  - name: Avg Order Value
    expr: SUM(source.net_sales) / COUNT(DISTINCT source.order_key)
  - name: Avg Selling Price
    expr: SUM(source.net_sales) / SUM(source.quantity)
  - name: Weighted Discount Rate
    expr: SUM(source.extended_price * source.discount) / SUM(source.extended_price)
  - name: Return Rate
    expr: SUM(CASE WHEN source.return_flag = 'R' THEN source.quantity ELSE 0 END) / SUM(source.quantity)
  - name: On-Time Delivery Rate
    expr: SUM(CASE WHEN source.receipt_date <= source.commit_date THEN 1 ELSE 0 END) / COUNT(1)
  - name: Avg Fulfillment Lead Time
    expr: AVG(datediff(source.receipt_date, source.order_date))
"""

# Metric View 2 — supply / sourcing semantics over fct_part_supply + part/supplier/geo dims.
# fct_part_supply carries the current-version part/supplier surrogate keys; join on them.
supply_metrics_yaml = f"""
version: 0.1
source: {g}.fct_part_supply
joins:
  - name: prod
    source: {g}.dim_part
    on: source.part_sk = prod.part_sk
  - name: sup
    source: {g}.dim_supplier
    on: source.supplier_sk = sup.supplier_sk
dimensions:
  - name: Brand
    expr: prod.brand
  - name: Part Type
    expr: prod.part_type
  - name: Manufacturer
    expr: prod.manufacturer
  - name: Supplier
    expr: sup.supplier_name
  - name: Supplier Nation
    expr: sup.nation_name
  - name: Supplier Region
    expr: sup.region_name
measures:
  - name: Inventory Value
    expr: SUM(source.available_qty * source.supply_cost)
  - name: Available Units
    expr: SUM(source.available_qty)
  - name: Avg Supply Cost
    expr: AVG(source.supply_cost)
  - name: Supplier Count
    expr: COUNT(DISTINCT source.supplier_key)
  - name: Part Count
    expr: COUNT(DISTINCT source.part_key)
  - name: Supply Records
    expr: COUNT(1)
"""


def create_metric_view(name, yaml_body):
    ddl = f"CREATE OR REPLACE VIEW {g}.{name} WITH METRICS LANGUAGE YAML AS $$\n{yaml_body.strip()}\n$$"
    spark.sql(ddl)
    print(f"  created metric view {g}.{name}")


print(f"Creating metric views in {g} ...")
create_metric_view("tpch_sample_sales_metrics", sales_metrics_yaml)
create_metric_view("tpch_sample_supply_metrics", supply_metrics_yaml)
print("Done.")


## Example: same metric, both ways

```sql
-- Pre-aggregated MV (fixed grain)
SELECT sales_month, market_segment, net_sales
FROM IDENTIFIER(:catalog || '.' || :schema_namespace || '_gold' || :logical_env || '.fct_sales_by_month_segment');

-- Metric View (any grain, define-once semantics)
SELECT `Order Month`, `Market Segment`,
       MEASURE(`Net Sales`), MEASURE(`On-Time Delivery Rate`), MEASURE(`Avg Order Value`)
FROM IDENTIFIER(:catalog || '.' || :schema_namespace || '_gold' || :logical_env || '.mv_sales_metrics')
GROUP BY 1, 2;
```